In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').getOrCreate()

In [20]:
# Sample data
data = [('James', 'Sales', 'NY', 9000, 34),
        ('Alicia', 'Sales', 'NY', 8600, 56),
        ('Robert', 'Sales', 'CA', 8100, 30),
        ('Lisa', 'Finance', 'CA', 9000, 24),
        ('Deja', 'Finance', 'CA', 9900, 40),
        ('Sugie', 'Finance', 'NY', 8300, 36),
        ('Ram', 'Finance', 'NY', 7900, 53),
        ('Kyle', 'Marketing', 'CA', 8000, 25),
        ('Reid', 'Marketing', 'NY', 9100, 50)]
df = spark.createDataFrame(data, ['Name', 'Department', 'State', 'Salary', 'Age'])
df.show()

+------+----------+-----+------+---+
|  Name|Department|State|Salary|Age|
+------+----------+-----+------+---+
| James|     Sales|   NY|  9000| 34|
|Alicia|     Sales|   NY|  8600| 56|
|Robert|     Sales|   CA|  8100| 30|
|  Lisa|   Finance|   CA|  9000| 24|
|  Deja|   Finance|   CA|  9900| 40|
| Sugie|   Finance|   NY|  8300| 36|
|   Ram|   Finance|   NY|  7900| 53|
|  Kyle| Marketing|   CA|  8000| 25|
|  Reid| Marketing|   NY|  9100| 50|
+------+----------+-----+------+---+



### Group by single column

In [21]:
df.groupBy(df.Department).count().show()

+----------+-----+
|Department|count|
+----------+-----+
|     Sales|    3|
|   Finance|    4|
| Marketing|    2|
+----------+-----+



In [22]:
df.groupBy(df.Department).avg('Salary').show()

+----------+-----------------+
|Department|      avg(Salary)|
+----------+-----------------+
|     Sales|8566.666666666666|
|   Finance|           8775.0|
| Marketing|           8550.0|
+----------+-----------------+



In [47]:
# Multiple aggregations using agg()
from pyspark.sql.functions import count, sum, avg, max, min
df.groupBy('Department').agg(
    count('Name').alias('employee_count'),
    sum('Salary').alias('total_salary'),
    avg('Salary').alias('avg_salary'),
    max('Salary').alias('max_salary'),
    min('Salary').alias('min_salary')
).show()

+----------+--------------+------------+-----------------+----------+----------+
|Department|employee_count|total_salary|       avg_salary|max_salary|min_salary|
+----------+--------------+------------+-----------------+----------+----------+
|     Sales|             3|       25700|8566.666666666666|      9000|      8100|
|   Finance|             4|       35100|           8775.0|      9900|      7900|
| Marketing|             2|       17100|           8550.0|      9100|      8000|
+----------+--------------+------------+-----------------+----------+----------+



Group by multiple columns

In [46]:
df.groupBy('Department', 'Name').agg(
    sum('Salary').alias('total_salary'),
    count('*').alias('record_count')
).show()

+----------+------+------------+------------+
|Department|  Name|total_salary|record_count|
+----------+------+------------+------------+
|   Finance|  Lisa|        9000|           1|
|     Sales|Alicia|        8600|           1|
|     Sales| James|        9000|           1|
|     Sales|Robert|        8100|           1|
|   Finance| Sugie|        8300|           1|
| Marketing|  Kyle|        8000|           1|
| Marketing|  Reid|        9100|           1|
|   Finance|   Ram|        7900|           1|
|   Finance|  Deja|        9900|           1|
+----------+------+------------+------------+



### Pivot

In [44]:
agg_df = df.groupBy(df.Department, df.State).agg(sum(df.Salary).alias('salary_sum'))
agg_df.show()

+----------+-----+----------+
|Department|State|salary_sum|
+----------+-----+----------+
|     Sales|   CA|      8100|
|   Finance|   CA|     18900|
|     Sales|   NY|     17600|
|   Finance|   NY|     16200|
| Marketing|   NY|      9100|
| Marketing|   CA|      8000|
+----------+-----+----------+



In [49]:
pivot_df = agg_df.groupBy('Department').pivot('State').sum('salary_sum')
pivot_df.show()

+----------+-----+-----+
|Department|   CA|   NY|
+----------+-----+-----+
|     Sales| 8100|17600|
|   Finance|18900|16200|
| Marketing| 8000| 9100|
+----------+-----+-----+



### Unpivot

In [68]:
# Using Stack
# STACK(2, "CA", CA, "NY", NY)
#       ↑   ↑     ↑   ↑    ↑
#       │   │     │   │    └─ Value from NY column
#       │   │     │   └────── Label for second row
#       │   │     └────────── Value from CA column
#       │   └──────────────── Label for first row
#       └──────────────────── Create 2 rows per input row

pivot_df.createOrReplaceTempView('pivot_tbl')
spark.sql('SELECT Department, STACK(2, "Canada", CA, "New York", NY) AS (State, Salary) FROM pivot_tbl').show()

+----------+--------+------+
|Department|   State|Salary|
+----------+--------+------+
|     Sales|  Canada|  8100|
|     Sales|New York| 17600|
|   Finance|  Canada| 18900|
|   Finance|New York| 16200|
| Marketing|  Canada|  8000|
| Marketing|New York|  9100|
+----------+--------+------+



In [67]:
# Using Unpivot
spark.sql('SELECT Department, State, Salary FROM pivot_tbl UNPIVOT (Salary FOR State IN (CA, NY))').show()

+----------+-----+------+
|Department|State|Salary|
+----------+-----+------+
|     Sales|   CA|  8100|
|     Sales|   NY| 17600|
|   Finance|   CA| 18900|
|   Finance|   NY| 16200|
| Marketing|   CA|  8000|
| Marketing|   NY|  9100|
+----------+-----+------+

